# Exploracao do PlantVillage

Auditoria inicial do dataset PlantVillage usando apenas as imagens em `raw/color/`, sem extrair o ZIP e sem preparar split ou treinamento.

In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

In [ ]:
if IN_COLAB:
    BASE_DIR = Path("/content/drive/MyDrive/TCC")
else:
    current_dir = Path.cwd().resolve()
    BASE_DIR = current_dir.parent if current_dir.name == "notebooks" else current_dir

DATA_DIR = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"
SRC_DIR = BASE_DIR / "src"

ZIP_PATH = DATA_DIR / "data.zip"
LEAF_MAP_PATH = DATA_DIR / "leaf_grouping" / "leaf-map.json"
OUTPUT_CSV = RESULTS_DIR / "plantvillage_metadata_raw_color.csv"

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Base:", BASE_DIR)
print("ZIP:", ZIP_PATH)
print("Leaf map:", LEAF_MAP_PATH)
print("Resultados:", RESULTS_DIR)

## Arquivos oficiais

In [ ]:
try:
    from huggingface_hub import hf_hub_download
except ImportError:
    %pip install -q huggingface_hub
    from huggingface_hub import hf_hub_download

In [ ]:
if not ZIP_PATH.exists():
    ZIP_PATH = Path(
        hf_hub_download(
            repo_id="mohanty/PlantVillage",
            filename="data.zip",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
        )
    )

if not LEAF_MAP_PATH.exists():
    LEAF_MAP_PATH = Path(
        hf_hub_download(
            repo_id="mohanty/PlantVillage",
            filename="leaf_grouping/leaf-map.json",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
        )
    )

print("data.zip existe:", ZIP_PATH.exists())
print("leaf-map.json existe:", LEAF_MAP_PATH.exists())

if ZIP_PATH.exists():
    print(f"Tamanho do data.zip: {ZIP_PATH.stat().st_size / (1024 ** 3):.2f} GB")

## Metadados

In [ ]:
import importlib.util
import sys
import urllib.request

module_file = SRC_DIR / "plantvillage_audit.py"
required_markers = (
    "FALLBACK_PREFIX",
    "leaf_id_source",
    "origem_leaf_id",
    "numero_identificadores_agrupamento_unicos",
    "maiores_grupos_leaf_id",
    "maiores_grupos_fallback",
    "fallbacks_multiclasse",
)

module_text = module_file.read_text(encoding="utf-8") if module_file.exists() else ""
if not all(marker in module_text for marker in required_markers):
    SRC_DIR.mkdir(parents=True, exist_ok=True)
    module_url = "https://raw.githubusercontent.com/murilodc/plant-disease-classification/main/src/plantvillage_audit.py"
    module_text = urllib.request.urlopen(module_url).read().decode("utf-8")
    module_file.write_text(module_text, encoding="utf-8")
    print("Modulo atualizado em:", module_file)

module_text = module_file.read_text(encoding="utf-8")
missing_markers = [marker for marker in required_markers if marker not in module_text]
if missing_markers:
    raise RuntimeError(f"Modulo PlantVillage ainda esta desatualizado: {module_file}")

spec = importlib.util.spec_from_file_location("plantvillage_audit_runtime", module_file)
if spec is None or spec.loader is None:
    raise ImportError(f"Nao foi possivel carregar o modulo: {module_file}")

pv_audit = importlib.util.module_from_spec(spec)
sys.modules["plantvillage_audit_runtime"] = pv_audit
spec.loader.exec_module(pv_audit)

audit_metadata = pv_audit.audit_metadata
build_metadata_dataframe = pv_audit.build_metadata_dataframe
save_metadata_csv = pv_audit.save_metadata_csv

print("Modulo carregado de arquivo:", pv_audit.__file__)

fallback_test = pv_audit.resolve_leaf_id("x.JPG", "Classe___Teste", {})
if fallback_test.get("leaf_id") != "fallback_x" or fallback_test.get("leaf_id_source") != "fallback":
    raise RuntimeError(f"Modulo PlantVillage desatualizado: {pv_audit.__file__}")

metadata = build_metadata_dataframe(
    zip_path=ZIP_PATH,
    leaf_map_path=LEAF_MAP_PATH,
)

print("Formato do DataFrame:", metadata.shape)
metadata.head()

## Resumo da auditoria

In [ ]:
auditoria = audit_metadata(metadata)

print("Chaves da auditoria:", list(auditoria))

auditoria["resumo"]

## Origem dos identificadores

In [ ]:
auditoria["origem_leaf_id"]

In [ ]:
associadas_leaf_map = metadata.loc[metadata["leaf_id_source"].eq("leaf-map")]
fallbacks = metadata.loc[metadata["leaf_id_source"].eq("fallback")]

print("Imagens associadas pelo leaf-map:", len(associadas_leaf_map))
print("Imagens usando fallback:", len(fallbacks))

## Status da associacao

In [ ]:
auditoria["status_leaf_id"]

## Maiores grupos leaf_id

In [ ]:
auditoria["maiores_grupos_leaf_id"]

## Maiores grupos fallback

In [ ]:
auditoria["maiores_grupos_fallback"]

## Colisoes de fallback

In [ ]:
resumo = auditoria["resumo"].iloc[0]

print("Fallbacks com mais de uma imagem:", resumo["fallbacks_com_mais_de_uma_imagem"])
print("Fallbacks em mais de uma classe:", resumo["fallbacks_em_mais_de_uma_classe"])

## Grupo fallback_r

In [ ]:
fallback_r = auditoria["fallback_r_imagens"]

print("Imagens no grupo fallback_r:", len(fallback_r))
if len(fallback_r) > 0:
    print("Classes no grupo fallback_r:", fallback_r["quantidade_classes_no_grupo"].iloc[0])
else:
    print("Classes no grupo fallback_r: 0")

fallback_r.head(20)

In [ ]:
auditoria["fallback_r_imagens_por_classe"]

## Fallbacks em mais de uma classe

In [ ]:
auditoria["fallbacks_multiclasse"]

## Imagens por classe

In [ ]:
auditoria["imagens_por_classe"]

## Salvar metadados

In [ ]:
csv_path = save_metadata_csv(metadata, OUTPUT_CSV)

print("CSV salvo em:", csv_path)
print("Arquivo existe:", csv_path.exists())